# Load Libraries and Data

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors

from shapely.geometry import Point
import statsmodels.formula.api as smf

from particles_path import *
from is_in_earth_shadow import *


#Load the cleaned data
clean_rad = pd.read_excel("clean_data_kp.xlsx")

#### Create geodataframe for geopandas

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

geometry = [Point(xy) for xy in zip(clean_rad["lon"], clean_rad["lat"])]
geo_rad = gpd.GeoDataFrame(clean_rad, geometry=geometry, crs="EPSG:4326")

### Define a constant colormap to use for all visuals

Default: `managua` 

`managua` is very good for highlighting all points on the graph.  
Change to `Reds` if you would rather only easily see extreme high values. All low values become very hard to see on the light background

In [ ]:
COLOR = "managua"

### Define a constant max and min for kp colorbars

In [ ]:
KP_MAX = 5.667
KP_MIN = 0

### Create a new geodataframe where all kp values are floored

In [ ]:
floored = geo_rad.copy()
floored[["kp", "kp_lag3", "kp_lag6", "kp_lag24", "kp_lag48"]] = np.floor(floored[["kp", "kp_lag3", "kp_lag6", "kp_lag24", "kp_lag48"]])

# Initial Look at Kp

##### KP signficiance:

https://theaurorazone.com/nuts-about-kp/

|Kp Scale|Auroral Activity|Frequency|
|---|---|---|
|0|Quiet|
|1|Quiet|A range between Kp1 to Kp3|
|2|Quiet|is most frequently observed|
|3|Unsettled|
|4|Active|
|5|Minor Storm|900 Days per 11 Years|
|6|Moderate Storm|360 Days per 11 Years|
|7|Strong Storm|130 Days per 11 Years|
|8|Severe Storm|60 Days per 11 Years|
|9|Extreme Storm|4 Days per 11 Years|


In [ ]:
clean_rad["kp"].describe()

## How many instances of each kp value are there?

In [ ]:
clean_rad.groupby("kp")["kp"].count()

## How many data points are in each bin?

In [ ]:
floored.groupby("kp")["kp"].count().plot(kind="bar")
plt.xlabel('Kp Index')
plt.ylabel('Counts')
plt.show()

# Compare Kp index to xray radiation

## Bin X-rays by Kp index

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
axes = axes.flatten()


for i in range(6):
    subset = floored[floored["kp"] == i]

    world.plot(ax=axes[i], color="lightgrey")
    subset.plot(ax=axes[i],
                markersize=1,
                column="xray0_ps",
                alpha=0.75,
                cmap=COLOR,
                legend=True)
    axes[i].set_title(f"Kp between {i} and {i+1}")

fig.suptitle("xray0_ps binned by kp", fontsize="xx-large")
plt.show()

### Analysis of X-ray binned by kp
There are high and low values in all Kp bins. I expected the values to be higher in the higher bins, but that doesn't appear to be the case. I will investigate this further

## Highlight Kp over 5 bcause it's when magnetic storms start

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["kp"] >= 5].plot(ax=ax,
                                 column="xray0_ps",
                                 markersize=20,
                                 alpha=0.75,
                                 cmap=COLOR,
                                 legend=True
)
ax.set_title("Kp >= 5")
plt.show()

X-rays do not appear to be consistently high during when `kp >= 5`

## Highlight Kp less than 3 because that's when the atmosphere is considered quiet

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["kp"] < 3].plot(ax=ax,
                                 column="xray0_ps",
                                 markersize=20,
                                 alpha=0.75,
                                 cmap=COLOR,
                                 legend=True
)
ax.set_title("Kp < 3")
plt.show()

X-rays do not appear to be consistently low when `kp < 3`

## View all `xray0_ps >= 10,000` colored by Kp index

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["xray0_ps"] >= 10000].plot(ax=ax,
                                           column="kp",
                                           markersize=20,
                                           alpha=0.75,
                                           cmap=COLOR,
                                           legend=True,
                                           legend_kwds={"label": "Kp index"},
                                           vmax = KP_MAX,
                                           vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All xrays per second >= 10000 colored by Kp")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["xray0_ps"] <= 1].plot(ax=ax,
                                          column="kp",
                                          markersize=20,
                                          alpha=0.75,
                                          cmap=COLOR,
                                          legend=True,
                                          legend_kwds={"label": "Kp index"},
                                          vmax = KP_MAX,
                                          vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All xrays per second <= 1 colored by Kp")
plt.show()

### Analysis of coloring by Kp
It appears that we find all Kp values from 0 to 5.667 show up with high and low values. As of yet, Kp doesn't appear to be making much of a difference in `xray0_ps`.

## Boxplots of bins

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)

floored.boxplot(column="xray0_ps", 
                by="kp", 
                ax=ax,
                patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6),
                medianprops=dict(color="black", linewidth=2),
                flierprops=dict(marker="o", markersize=2, alpha=0.3))

plt.suptitle("")  # removes the automatic "Boxplot grouped by kp" title
ax.set_yscale("log")
ax.set_title("xray0_ps by kp")
ax.set_ylabel("Particle Counts Per Second (Log)")
plt.show()

## Analysis of boxplots
The median X-ray values of Kps in the 2, 3, and 4 bins are highest. I expected kp 5 to have the hightest, but it did not. Kp 5 has a much smaller sample size of 123 (compared to 2502 and 1201 for kps 3 and 4 respectively), so that may skew results. Bins 0 and 1 are very low compared to 2, 3, and 4, and that was expected.

# Compre Kp index to proton radiation

## Bin protons by Kp index

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
axes = axes.flatten()


for i in range(6):
    subset = floored[floored["kp"] == i]
    
    world.plot(ax=axes[i], color="lightgrey")
    subset.plot(ax=axes[i],
                markersize=1,
                column="proton0_ps",
                alpha=0.75,
                cmap=COLOR,
                legend=True)
    axes[i].set_title(f"Kp between {i} and {i+1}")

fig.suptitle("proton0_ps binned by kp", fontsize="xx-large")
plt.show()

### Analysis of X-ray binned by kp
Just like with X-rays, there are high and low values in all Kp bins. From my minimal understanding of proton vs X-ray radiation, this is less surprising, but it is still worth checking

## View all `proton0_ps >= 10,000` colored by Kp index

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["proton0_ps"] >= 10000].plot(ax=ax,
                                           column="kp",
                                           markersize=20,
                                           alpha=0.75,
                                           cmap=COLOR,
                                           legend=True,
                                           legend_kwds={"label": "Kp index"},
                                           vmax = KP_MAX,
                                           vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All protons per second >= 10000 colored by Kp")
plt.show()

## View all `proton0_ps <= 2,000` colored by Kp index

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["proton0_ps"] <= 1].plot(ax=ax,
                                          column="kp",
                                          markersize=20,
                                          alpha=0.75,
                                          cmap=COLOR,
                                          legend=True,
                                          legend_kwds={"label": "Kp index"},
                                          vmax = KP_MAX,
                                          vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All protons per second <= 1 colored by Kp")
plt.show()

### Analysis of coloring by Kp
It appears that we find all Kp values from 0 to 5.667 show up with high and low values. As of yet, Kp doesn't appear to be making much of a difference in `proton0_ps`.

## Boxplots of bins

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)

floored.boxplot(column="proton0_ps", 
                by="kp", 
                ax=ax,
                patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6),
                medianprops=dict(color="black", linewidth=2),
                flierprops=dict(marker="o", markersize=2, alpha=0.3))

plt.suptitle("")  # removes the automatic "Boxplot grouped by kp" title
ax.set_yscale("log")
ax.set_title("proton0_ps by kp")
ax.set_ylabel("Particle Counts Per Second (Log)")
plt.show()

### Analysis of boxplots
Kp 5 was lower here just like in `xray0_ps`. Other than that, all medians are fairly close to each other

# Compare Kp index to electron radiation

## Bin electrons by Kp index

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
axes = axes.flatten()


for i in range(6):
    subset = floored[floored["kp"] == i]

    world.plot(ax=axes[i], color="lightgrey")
    subset.plot(ax=axes[i],
                markersize=1,
                column="electron0_ps",
                alpha=0.75,
                cmap=COLOR,
                legend=True)
    axes[i].set_title(f"Kp between {i} and {i+1}")

fig.suptitle("electron0_ps binned by kp", fontsize="xx-large")
plt.show()

### Analysis of electrons binned by kp
Electrons tend to say pretty low across the board. There is no specific bin where it is consistently elevated. We have elevated values in all bins, but most values in all bins are low

## View all `electron0_ps >= 10,000` colored by Kp index

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["electron0_ps"] >= 10000].plot(ax=ax,
                                          column="kp",
                                          markersize=20,
                                          alpha=0.75,
                                          cmap=COLOR,
                                          legend=True,
                                          legend_kwds={"label": "Kp index"},
                                          vmax = KP_MAX,
                                          vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All electrons per second >= 10000 colored by Kp")
plt.show()

## View all `electron0_ps <= 2,000` colored by Kp index

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["electron0_ps"] <= 1].plot(ax=ax,
                                          column="kp",
                                          markersize=20,
                                          alpha=0.75,
                                          cmap=COLOR,
                                          legend=True,
                                          legend_kwds={"label": "Kp index"},
                                          vmax = KP_MAX,
                                          vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All electrons per second <= 1 colored by Kp")
plt.show()

### Analysis of coloring by Kp
We only have two points over 10,000 electrons per second. Interestingly, those two points have very different Kps of 2 and 4. Just like X-ray and proton, it Kp doesn't appear to have much of an effect on `electron0_ps`

## Boxplots of bins

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)

floored.boxplot(column="electron0_ps", 
                by="kp", 
                ax=ax,
                patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6),
                medianprops=dict(color="black", linewidth=2),
                flierprops=dict(marker="o", markersize=2, alpha=0.3))

plt.suptitle("")  # removes the automatic "Boxplot grouped by kp" title
ax.set_yscale("log")
ax.set_title("electron0_ps by kp")
ax.set_ylabel("Particle Counts Per Second (Log)")
plt.show()

### Analysis of boxplots
Kp 5 was way lower here, noticably lower than in `xray0_ps` or `proton0_ps`. I assume this is partially because of the sample size difference, but I cannot say for sure. Other than that, all medians are fairly close to each other

# Create a correlation matrix to check Kp's effect on radiation

In [ ]:
corr = clean_rad[["kp", "xray0_ps", "proton0_ps", "electron0_ps"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap=COLOR, vmin=-1, vmax=1, ax=ax)

ax.xaxis.tick_top()          # move tick marks + labels to the top
plt.show()

## Analysis of correlation matrix
All radiation columns have a very low correlation with Kp. This is consistent with what I've observed from the previous analysis in this file.

While the number is incredibly low, it is interesting that `kp` and `electron0_ps` show an inverse correlation.

Also, `proton0_ps` and `electron0_ps` have a very high correlation with each other. This is unsurprising from what's been seen in previous analysis.

# Does the effect of a geomagnetic storm linger?

## What is the max kp value of each lagged kp column?

In [ ]:
clean_rad[["kp", "kp_lag3", "kp_lag6", "kp_lag12", "kp_lag24", "kp_lag48", "kp_lag72"]].max()

In [ ]:
clean_rad[clean_rad["kp_lag48"] > 5.667]

In [ ]:
clean_rad[clean_rad["kp_lag12"] > 5.667]["group"]

`kp_lag48` has a max value of 6.667 which is higher than the visuals I have created can handle. I will plot this individually  
Similarly, `kp_lag12` has a max value of 8. Turns out, this is for the entirety of group 1. I will plot that separately as well

## How does lagging kp affect xray?

In [ ]:
for lag in ["kp_lag3", "kp_lag6", "kp_lag12", "kp_lag24", "kp_lag48", "kp_lag72"]:
    fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
    axes = axes.flatten()

    for i in range(6):
        subset = floored[floored[lag] == i]
        
        world.plot(ax=axes[i], color="lightgrey")
        subset.plot(ax=axes[i],
                    markersize=1,
                    column="xray0_ps",
                    alpha=0.75,
                    cmap=COLOR,
                    legend=True)
        axes[i].set_title(f"Kp between {i} and {i+1}")
        axes[i].set_aspect("equal", adjustable="box")
    fig.suptitle(f"xray0_ps binned by {lag}", fontsize="xx-large")
    plt.show()

## How does lagging kp affect proton?

In [ ]:
for lag in ["kp_lag3", "kp_lag6", "kp_lag24", "kp_lag48"]:
    fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
    axes = axes.flatten()

    for i in range(6):
        subset = floored[floored[lag] == i]
        
        world.plot(ax=axes[i], color="lightgrey")
        subset.plot(ax=axes[i],
                    markersize=1,
                    column="proton0_ps",
                    alpha=0.75,
                    cmap=COLOR,
                    legend=True)
        axes[i].set_title(f"Kp between {i} and {i+1}")
    fig.suptitle(f"proton0_ps binned by {lag}", fontsize="xx-large")
    plt.show()

## How does lagging kp affect electron?

In [ ]:
for lag in ["kp_lag3", "kp_lag6", "kp_lag24", "kp_lag48"]:
    fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
    axes = axes.flatten()

    for i in range(6):
        subset = floored[floored[lag] == i]
        
        world.plot(ax=axes[i], color="lightgrey")
        subset.plot(ax=axes[i],
                    markersize=1,
                    column="electron0_ps",
                    alpha=0.75,
                    cmap=COLOR,
                    legend=True)
        axes[i].set_title(f"Kp between {i} and {i+1}")
    fig.suptitle(f"electron0_ps binned by {lag}", fontsize="xx-large")
    plt.show()

## Analysis of comparing all three sensors to lagged kp values

All four of the lagged kp values show the same trend as our original kp. All three sensors show spikes across all 6 kp bins, showing a low correlation yet again.

## What about `kp_lag48` values greater than 5.667?
All of these values will have a floored value of 6, so I will used `floored` instead of `geo_rad`

In [ ]:
subset = floored[floored["kp_lag48"] == 6]

for col in ["xray0_ps", "proton0_ps", "electron0_ps"]:
    fig, ax = plt.subplots(figsize=(14,5), constrained_layout=True)

    world.plot(ax=ax, color="lightgrey")
    subset.plot(ax=ax,
                markersize=20,
                column=col,
                alpha=0.75,
                cmap=COLOR,
                legend=True)
    
    ax.set_title(f"{col} with kp_lag48 >= 6")
    plt.show()

### Analysis of `kp_lag48` values greater than 5.667

These are not consistently high either. There are significantly more low values for all three sensors than high values despite a kp value this high being considered a moderate storm. This is further leading me to believe that potential lasting effects of a storm also don't have much correlation with our particle counts

## What about Group 1's `kp_lag12` value of 8?

In [ ]:
subset = floored[floored["group"] == 1]

for col in ["xray0_ps", "proton0_ps", "electron0_ps"]:
    fig, ax = plt.subplots(figsize=(14,5), constrained_layout=True)

    world.plot(ax=ax, color="lightgrey")
    subset.plot(ax=ax,
                markersize=20,
                column=col,
                alpha=0.75,
                cmap=COLOR,
                legend=True)
    
    ax.set_title(f"{col} with kp_lag12 == 8")
    plt.show()

### Analysis of `kp_lag12` values equal to 8

These are not consistently high either. You would expect that when kp is equal to 8, but it's not there. There are significantly more low values for all three sensors than high values despite a kp value this high being considered a severe storm. This is further leading me to believe that potential lasting effects of a storm also don't have much correlation with our particle counts

# Correlation matrix for lagged Kp's effect on radiation

In [ ]:
corr = clean_rad[["kp_lag3", "kp_lag6", "kp_lag12", "kp_lag24", "kp_lag48", "kp_lag72", "xray0_ps", "proton0_ps", "electron0_ps"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, cmap=COLOR, vmin=-1, vmax=1, ax=ax)

ax.xaxis.tick_top()          # move tick marks + labels to the top
ax.tick_params(axis='x', labelsize=11)
plt.show()

### Analysis of lagged kp correlation matrix

Just as expecte, there is not a very high correlation between any of the lagged kp values and any of our particle sensors. The closest we get is a 0.14 correlation between `xray0_ps` and `kp_lag24`, and that is nowhere near high enough to conclude correlation. I believe it is safe to say that, at least for our data, there is no correlation between kp index and X-ray, proton, or electron radiation at this altutude

# Does AP index have any correlation with radiation?
AP indext is very similar to Kp index, with the major differences being that AP is only measured once a day, and AP is linear. I am curious if either of these differences show any different trends.

## Initial Look at Kp

In [ ]:
clean_rad["Ap"].describe()

### How many instances of each Ap value are there?

In [ ]:
clean_rad.groupby("Ap")["Ap"].count()

## Bin the Ap values by their 10s place value

In [ ]:
geo_rad["Ap_bins"] = (np.floor(geo_rad["Ap"] / 10) * 10).astype(int)

### How many data points are in each bin?

In [ ]:
geo_rad.groupby("Ap_bins")["Ap_bins"].count().plot(kind="bar")
plt.xlabel('Ap Index')
plt.ylabel('Counts')
plt.show()

## Make a heatmap of particle counts binned by Ap value

In [ ]:
for sensor in ["xray0_ps", "proton0_ps", "electron0_ps"]:
    fig, axes = plt.subplots(2, 3, figsize=(20,6))
    axes = axes.flatten()

    for i in range(0, 5):
        subset = geo_rad[geo_rad["Ap_bins"] == i*10]
            
        world.plot(ax=axes[i], color="lightgrey")
        subset.plot(ax=axes[i],
                    markersize=1,
                    column=sensor,
                    alpha=0.75,
                    cmap=COLOR,
                    legend=True)
        axes[i].set_title(f"Ap between {i*10} and {i*10+10}")
        axes[i].set_aspect("equal", adjustable="box")
    fig.suptitle(f"{sensor} binned by Ap Index", fontsize="xx-large")
    axes[-1].set_visible(False)
    plt.show()

### Analysis of Ap heatmap
This all looks very similar to Kp index. All sensors have peaks and valleys in each bin, with no bin having values consistently high or low. The only new thing that may be potentially worth noting is that all Ap values between 30 and 40 appear in the northern hemisphere. This could mean they all occured in March when we had no southern hemisphere data.

## When were Ap values between 30 and 40 observed?

In [ ]:
subset = geo_rad[geo_rad["Ap_bins"] == 30]

subset.groupby("month")["Ap_bins"].count().plot(kind="bar")
plt.title("When were Ap values between 30 and 40 observed?")
plt.ylabel("Observation Counts")
plt.show()

It looks like a majority of our data with Ap betwen 30 and 40 occur in March, while the remaining ~250 just happened to be in the northern hemisphere as well

## Boxplots of Ap bins by sensor

In [ ]:
for sensor in ["xray0_ps", "proton0_ps", "electron0_ps"]:
    fig, ax = plt.subplots(figsize=(14, 9), constrained_layout=True)

    geo_rad.boxplot(column=sensor, 
                    by="Ap_bins", 
                    ax=ax,
                    patch_artist=True,
                    boxprops=dict(facecolor="steelblue", alpha=0.6),
                    medianprops=dict(color="black", linewidth=2),
                    flierprops=dict(marker="o", markersize=2, alpha=0.3))

    plt.suptitle("")  # removes the automatic "Boxplot grouped by kp" title
    ax.set_yscale("log")
    ax.set_title(f"{sensor} by Ap")
    ax.set_ylabel("Particle Counts Per Second (Log)")
    plt.show()

### Analysis of boxplots

It looks like the protons and electrons are higher in the Ap 40 bin, but it's hard to say how much of an impact that has when we have similar sample size issues to kp index. It might be worth remembering, though.

## Correlation Matrix

In [ ]:
corr = clean_rad[["kp", "Ap", "xray0_ps", "proton0_ps", "electron0_ps"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap=COLOR, vmin=-1, vmax=1, ax=ax)

ax.xaxis.tick_top()          # move tick marks + labels to the top
plt.show()

### Analysis of correlation matrix with Ap index
The correlation between kp and our sensors and ap and our sensors are very, very close to each other. It looks like Ap doesn't have much correlation with any sensor either.

# Final Analysis of Kp and Ap
I am very comfortable saying that both Kp index and Ap index do not have a noticable effect on our radiation counts. Maybe we would come to a different conclusion if we had more data, but we can't say that with confidence. We have proof enough from numerous heatmaps, boxplots, and correlation matrices analyzing Kp index, the after affects of Kp index, and Ap index to comfortably say that these measures of geomagnetic disturbance do not have significant correlation with our particle counts for any sensor